<div style="background:linear-gradient(90deg,#C8102E 0%,#7A0019 100%); color:white; padding:22px 28px; border-radius:8px">
  <div style="font-size:0.9em; letter-spacing:2px; opacity:0.85">NOTEBOOK 03 · EXPLORE · COMPLETED</div>
  <div style="font-size:2.0em; font-weight:700; margin-top:4px">🔎 Exploratory data analysis</div>
  <div style="font-size:1.1em; margin-top:8px; max-width:880px; opacity:0.95">
    Get to know the cohort in a few cells, and surface the exact gap that justifies the NLP work
    that follows.
  </div>
</div>

## How easy is EDA on Databricks?

Every query below returns a Spark DataFrame. Click **`+` → Visualization** on any result to chart
it: no matplotlib, no export. We explore in three passes:
1. **Who are these patients?** demographics, stage, menopausal status
2. **What do the biomarkers look like?** HER2 / ER / PR distribution
3. **Where do the biomarkers live?** the structured-vs-notes gap (the punchline you'll quantify)

In [0]:
%run ./_config

# ⚙️ Configuration: Clinical Trial Pre-Screening (PRE-BUILT)

<div style="background:#f4f6f9; border-left:6px solid #C8102E; padding:14px 18px; border-radius:4px; font-size:0.95em">
This is the <b>companion config notebook</b>. It is <b>pre-built; you do not edit it</b>.
Every other notebook starts with <code>%run ./_config</code> so they all share one
catalog / schema / warehouse and the same read-only OMOP source.<br>
Just set the widgets at the top of <code>00_START_HERE</code> (matching your
bundle's <code>client_catalog</code> / <code>client_schema</code> / <code>warehouse_id</code>
/ <code>source_schema</code>) and re-run.
</div>

Everything here is Unity-Catalog-scoped (no hive_metastore) and reads from widgets, no
hardcoded secrets.

ℹ️ Not creating catalog clinops_catalog (likely pre-provisioned / no CREATE CATALOG): (com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException) PERMISSION_D
✅ Writing to clinops_catalog.clinops_ml
   Reading read-only OMOP source from clinops_catalog.clinops_foundation
   SQL Warehouse: 0123456789abcdef


Helpers ready: fqn(), src(), show_md(), LLM_FAST, LLM_STRONG


## 1️⃣ Who are these patients? (try the suggested viz on each)

In [0]:
%sql
SELECT age_at_dx_years FROM silver_demographics WHERE age_at_dx_years IS NOT NULL;

age_at_dx_years
35
38
58
29
55
61
52
50
42
58


In [0]:
%sql
SELECT ajcc_stage, COUNT(*) AS patients
FROM silver_demographics WHERE ajcc_stage IS NOT NULL
GROUP BY ajcc_stage ORDER BY ajcc_stage;

ajcc_stage,patients
Stage I,52
Stage IIA,76
Stage IIB,74
Stage IIIA,59
Stage IIIB,21
Stage IV,18


In [0]:
%sql
SELECT COALESCE(menopausal_status, '(unknown)') AS menopausal_status, COUNT(*) AS patients
FROM silver_demographics GROUP BY ALL ORDER BY patients DESC;

menopausal_status,patients
Postmenopausal,206
(unknown),60
Premenopausal,34


In [0]:
%sql
SELECT 'HER2' AS marker, COALESCE(her2_status,'(missing)') AS status, COUNT(*) n FROM silver_biomarker_profile GROUP BY 2
UNION ALL SELECT 'ER', COALESCE(er_status,'(missing)'), COUNT(*) FROM silver_biomarker_profile GROUP BY 2
UNION ALL SELECT 'PR', COALESCE(pr_status,'(missing)'), COUNT(*) FROM silver_biomarker_profile GROUP BY 2
ORDER BY marker, status;

marker,status,n
ER,Negative,85
ER,Positive,155
HER2,Negative,121
HER2,Positive,119
PR,Negative,127
PR,Positive,113


## 3️⃣ Where do the biomarkers live? The gap 🎯

This is the whole reason for the NLP work. Classify every patient by **where** their biomarker
evidence exists: **structured** (rows in `measurement`) vs **note** (a pathology report in `note`).

<div style="background:#FFEBEE; border-left:6px solid #C62828; padding:12px 16px; border-radius:4px">
Patients with <b>a pathology note but NO structured measurement</b> are invisible to the silver
pipeline, and to every SQL cohort query. Real, potentially-eligible patients we would miss.
</div>

In [0]:
%sql
-- Two helper sets, then LEFT JOIN person to both and CASE on which side is NULL.
-- The 'notes-only' count is the number of patients a SQL-only pipeline silently misses.
WITH has_struct AS (
  SELECT DISTINCT person_id FROM measurement
  WHERE measurement_source_value IN ('HER2/neu','Estrogen receptor','Progesterone receptor')
),
has_note AS (
  SELECT DISTINCT person_id FROM note WHERE note_source_value = 'PATHOLOGY_REPORT'
)
SELECT
  CASE
    WHEN s.person_id IS NOT NULL AND n.person_id IS NOT NULL THEN 'both'
    WHEN s.person_id IS NULL     AND n.person_id IS NOT NULL THEN 'notes-only'
    WHEN s.person_id IS NOT NULL AND n.person_id IS NULL     THEN 'structured-only'
    ELSE 'neither'
  END AS biomarker_evidence,
  COUNT(*) AS patients
FROM person p
LEFT JOIN has_struct s ON p.person_id = s.person_id
LEFT JOIN has_note   n ON p.person_id = n.person_id
GROUP BY ALL
ORDER BY patients DESC;

biomarker_evidence,patients
both,180
structured-only,60
notes-only,60


Expected: **both ≈ 180, notes-only ≈ 60, structured-only ≈ 60.**

In [0]:
notes_only = spark.sql("""
  WITH has_struct AS (
    SELECT DISTINCT person_id FROM measurement
    WHERE measurement_source_value IN ('HER2/neu','Estrogen receptor','Progesterone receptor')
  )
  SELECT COUNT(*) n FROM note n
  WHERE n.note_source_value = 'PATHOLOGY_REPORT'
    AND n.person_id NOT IN (SELECT person_id FROM has_struct)
""").first()["n"]

total = spark.sql("SELECT COUNT(*) n FROM person").first()["n"]

show_md(f"""
<b>{notes_only} of {total} patients</b> ({100*notes_only/total:.0f}%) have their biomarker status
written <i>only</i> in a free-text pathology note. A structured cohort query returns <b>zero</b> of
them. Recovering them is the job of notebook 04 (ai_query) and notebook 05 (ClinicalBERT).
""")

60 of 300 patients (20%) have their biomarker status
written only in a free-text pathology note. A structured cohort query returns zero of
them. Recovering them is the job of notebook 04 (ai_query) and notebook 05 (ClinicalBERT).

In [0]:
%sql
WITH has_struct AS (
  SELECT DISTINCT person_id FROM measurement
  WHERE measurement_source_value IN ('HER2/neu','Estrogen receptor','Progesterone receptor')
)
SELECT person_id, note_title, note_text
FROM note
WHERE note_source_value = 'PATHOLOGY_REPORT'
  AND person_id NOT IN (SELECT person_id FROM has_struct)
ORDER BY person_id LIMIT 3;

person_id,note_title,note_text
181,Core Needle Biopsy — Pathology Report,"PATHOLOGY REPORT — CORE NEEDLE BIOPSY PATH-866538 | 2023-07-11 SPECIMEN: Right breast, vacuum-assisted biopsy INDICATION: Palpable axillary node, Right. Primary breast cancer suspected. HISTOLOGIC FINDINGS: Diagnosis: Invasive ductal carcinoma, NOS Nuclear Grade: 2/3 Estimated Tumor Size: 0.9 cm Lymphovascular Invasion: Not identified RECEPTOR / BIOMARKER STATUS: ER: Negative — no significant nuclear reactivity observed Progesterone receptor: non-reactive (<1%) HER2 (c-erbB-2): 3+ (strongly positive) Ki-67: 57% MOLECULAR SUBTYPE: HER2-positive (non-luminal) COMMENT: Clinical correlation recommended. These findings represent a biopsy diagnosis; final staging will require complete excision and axillary lymph node evaluation. Pathologist: Deborah Jones, M.D. | Meridian Cancer Center"
182,Surgical Pathology Report,"CONSULTATION SURGICAL PATHOLOGY Meridian Cancer Center — Department of Pathology Accession: PATH-386033 Outside MRN: MRN-000182 Date of Consultation: 05/11/2024 SUBMITTED MATERIAL: Right breast mass, ultrasound-guided biopsy CLINICAL SUMMARY: Right breast mass found on MRI; clinical stage T2N0. Outside pathology reviewed; confirmatory studies performed. HISTOPATHOLOGY: Invasive mammary carcinoma, no special type, Nottingham grade 2/3, tumor size approximately 3.9 cm. Not identified lymphovascular invasion. IHC / BIOMARKER RESULTS: Estrogen receptor status: POSITIVE (Allred 6/8; moderate staining) PR: Positive — moderate nuclear staining in 83% of cells HER2 (c-erbB-2): 3+ (strongly positive) Ki-67 labeling index: 58% INTERPRETATION: The above findings are consistent with HR+/HER2+ (luminal B-like, HER2-positive) breast carcinoma. Recommend multidisciplinary tumor board review for treatment planning. Attending Pathologist: Cody Gregory, M.D."
183,Pathology Consultation Report,"MERIDIAN CANCER CENTER DEPARTMENT OF PATHOLOGY — SURGICAL PATHOLOGY REPORT Accession No.: PATH-357494 Date Received: January 02, 2023 Date Reported: January 04, 2023 Patient MRN: MRN-000183 SPECIMEN SUBMITTED: Right breast, vacuum-assisted biopsy CLINICAL HISTORY: Right breast density with suspicious calcifications on mammography. GROSS DESCRIPTION: Received in formalin, labeled with patient MRN and specimen site. Core biopsy material, estimated 2 cores measuring up to 13 mm in greatest dimension. Submitted in entirety. MICROSCOPIC DESCRIPTION: Sections show invasive mammary carcinoma, no special type. Nottingham histologic grade 2/3 (tubular formation score 2, nuclear pleomorphism score 1, mitotic rate score 2; combined Nottingham score 6). Invasive tumor size estimated at 1.2 cm. Lymphovascular invasion: Not identified. IMMUNOHISTOCHEMISTRY: Estrogen receptor (ER): Positive (78% nuclear staining, Allred score 7/8) Progesterone receptor (PR): Positive (82% nuclear staining, Allred score 6/8) Her-2/neu: 0 (negative) Ki-67 proliferative index: 32% PATHOLOGIC SUBTYPE: HR+/HER2- (luminal A or B) DIAGNOSIS: Right breast, vacuum-assisted biopsy — Invasive mammary carcinoma, no special type Nuclear grade: 2/3 Tumor size (estimate): 1.2 cm Lymphovascular invasion: Not identified Receptor profile: HR+/HER2- (luminal A or B) Electronically signed by: Justin Cummings, MD 01/04/2023"


<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:12px 16px; border-radius:4px">
<b>Takeaway:</b> the notes phrase HER2/ER/PR a dozen ways (<i>"3+ by IHC"</i>, <i>"FISH ratio 3.1,
amplified"</i>, <i>"overexpression confirmed"</i>). No <code>LIKE</code> catches them all, but a
Foundation Model reads them like a clinician. That's next.
</div>

## ▶️ Next step
### → Open **[04_nlp_biomarker_extraction]($./04_nlp_biomarker_extraction)** to recover the notes-only patients with `ai_query`.